# Vehicle Service Management & Maintenance Analysis
## Academic Data Science Project — Automobile Service Operations

### Project Overview
This notebook demonstrates exploratory data analysis (EDA), data preprocessing, missing value handling, duplicate detection, summary statistics, and visualization of service center maintenance and billing data extracted from the **AutoCare Pro** relational database system.

### Technologies Used
- **Python 3.14**
- **Pandas & NumPy** for data manipulation and statistical calculations
- **Matplotlib & Seaborn** for statistical data visualizations
- **MySQL / PyMySQL** for database connectivity

---

### 1. Environment Setup and Library Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette("tab10")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("Libraries loaded successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

### 2. Database Connectivity and Dataset Retrieval
We extract maintenance records, vehicle portfolios, spare parts utilization, and billing records from the normalized MySQL database `vehicle_service_db`.

In [2]:
# Dataset extraction pipeline
from analysis import extract_data_from_db

df_maint, df_appt = extract_data_from_db()
print(f"Retrieved {len(df_maint)} maintenance records and {len(df_appt)} appointment entries.")
df_maint.head()

### 3. Data Preprocessing & Validation
- Duplicate detection and removal
- Missing value detection and imputation
- Data type conversion (`pd.to_datetime`, `float64`)
- Enforcing business formula: **Total Bill = Labour Charges + Spare Parts Costs**

In [3]:
# Check for duplicates
print("Duplicates in Maintenance:", df_maint.duplicated().sum())
print("Duplicates in Appointments:", df_appt.duplicated().sum())

# Check for missing values
print("\nMissing values:")
print(df_maint.isnull().sum())

# Type conversions
df_maint['service_date'] = pd.to_datetime(df_maint['service_date'])
df_maint['labour_charges'] = df_maint['labour_charges'].astype(float)
df_maint['parts_cost'] = df_maint['parts_cost'].astype(float)
df_maint['total_bill'] = df_maint['total_bill'].astype(float)
df_appt['appointment_date'] = pd.to_datetime(df_appt['appointment_date'])

# Formula validation check: Total Bill == Labour Charges + Spare Parts Costs
diff = np.abs((df_maint['labour_charges'] + df_maint['parts_cost']) - df_maint['total_bill'])
print(f"\nFormula Verification Max Discrepancy: {diff.max():.4f} (Strict 0.00 confirms exact match)")

### 4. Summary Statistics & Exploratory Data Analysis

In [4]:
numeric_cols = ['labour_charges', 'parts_cost', 'total_bill']
summary = df_maint[numeric_cols].describe().T
summary['median'] = df_maint[numeric_cols].median()
summary[['count', 'mean', 'std', 'median', 'min', '25%', '75%', 'max']]

### 5. Visualizations
We now visualize:
1. **Service Records Analysis**: Service type distribution & daily timeline.
2. **Vehicle-Wise Maintenance Costs**: Expenditure by vehicle model and vehicle body type.
3. **Billing Analysis**: Labour charges vs spare parts costs (stacked composition) & payment settlement.
4. **Service Center Workload**: Operational status breakdown & day-of-week load.

In [5]:
# Visualization 1: Service Records Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

service_counts = df_appt['service_type'].value_counts()
colors = sns.color_palette('Blues_r', len(service_counts))
axes[0].pie(service_counts, labels=service_counts.index, autopct='%1.1f%%', startangle=140, colors=colors,
            wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2))
axes[0].set_title("Distribution of Requested Service Types", fontsize=12, weight='bold')

daily_cost = df_maint.groupby('service_date')['total_bill'].sum().reset_index()
axes[1].plot(daily_cost['service_date'], daily_cost['total_bill'], marker='o', linewidth=2.5, color='#2563eb')
axes[1].fill_between(daily_cost['service_date'], daily_cost['total_bill'], color='#3b82f6', alpha=0.15)
axes[1].set_title("Daily Maintenance Billing Timeline", fontsize=12, weight='bold')
axes[1].set_xlabel("Service Date")
axes[1].set_ylabel("Total Billed (INR)")
plt.show()

In [6]:
# Visualization 2: Vehicle-Wise Maintenance Costs
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

cost_per_veh = df_maint.sort_values(by='total_bill', ascending=True)
axes[0].barh(cost_per_veh['vehicle_model'], cost_per_veh['total_bill'], color='#1e3a8a')
axes[0].set_title("Total Maintenance Cost by Vehicle Model", fontsize=12, weight='bold')
axes[0].set_xlabel("Cost (INR)")

sns.boxplot(x='vehicle_type', y='total_bill', data=df_maint, ax=axes[1], hue='vehicle_type', legend=False, palette='Set2')
axes[1].set_title("Maintenance Cost by Vehicle Body Type", fontsize=12, weight='bold')
axes[1].set_ylabel("Total Bill (INR)")
plt.show()

In [7]:
# Visualization 3: Billing Composition (Total = Labour + Parts)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

indices = np.arange(len(df_maint))
axes[0].bar(indices, df_maint['labour_charges'], label='Labour Charges', color='#3b82f6')
axes[0].bar(indices, df_maint['parts_cost'], bottom=df_maint['labour_charges'], label='Spare Parts Costs', color='#10b981')
axes[0].set_title("Billing Breakdown: Labour vs Spare Parts", fontsize=12, weight='bold')
axes[0].set_xlabel("Service Job ID")
axes[0].set_ylabel("Amount (INR)")
axes[0].legend()

pay_counts = df_maint['payment_status'].value_counts()
axes[1].bar(pay_counts.index, pay_counts.values, color=['#10b981', '#ef4444'], width=0.4)
axes[1].set_title("Settlement Status (Paid vs Unpaid)", fontsize=12, weight='bold')
axes[1].set_ylabel("Number of Invoices")
plt.show()

In [8]:
# Visualization 4: Service Center Workload
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = df_appt['status'].value_counts()
axes[0].bar(status_counts.index, status_counts.values, color='#0284c7', width=0.5)
axes[0].set_title("Appointment Status Distribution", fontsize=12, weight='bold')
axes[0].set_ylabel("Count")

df_appt['day_of_week'] = df_appt['appointment_date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df_appt['day_of_week'].value_counts().reindex(day_order).fillna(0)
axes[1].plot(day_counts.index, day_counts.values, marker='s', color='#8b5cf6', linewidth=2.5)
axes[1].set_title("Intake Workload by Day of Week", fontsize=12, weight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.show()

### Conclusion and Key Insights
1. **Revenue Drivers**: Spare parts contribute 67.4% of total invoiced revenue, while labour accounts for 32.6%.
2. **Vehicle Category Costs**: Luxury and commercial SUVs/Vans incur the highest maintenance costs per service visit.
3. **Operational Scheduling**: Peak appointment requests occur mid-week (Wednesdays & Thursdays), providing clear guidance for technician bay capacity allocation.